In [ ]:
# =================================================================
# Hybrid Deep Learning Model: Four class detection of heartbeats 
# =================================================================
# This script implements the R-peak detection algorithm described in my FYP report:
# ECG Signal Analysis Algorithm Empowered by AI, and represents its second part.

'''''
Author: Karim Nadjarian Hayek
Email: karimn.hayek@gmail.com
University Email: karimnh@purdue.edu
*****************************************************************************************************
COPYRIGHT: © 2024-2026 ALL RIGHTS RESERVED to the author.   

Neither this notebook nor any part may be reproduced or transmitted in any form or by any means,    
electronic or mechanical, including photocopying, microfilming, and recording, or by any information
storage or retrieval system, without prior permission in writing from the author, the company, or the
university.                   
*****************************************************************************************************
'''''

# =================================================================
# Required packages
# =================================================================
# - scipy.signal
# - numpy
# - math
# - matplotlib.pyplot
# - wfbd python (for reading test data and processing)
# - statsmodels.tsa.api (for exponential smoothing)
# - sklearn
# - imblearn
# - keras
# - seaborn

# Open source test data is taken from:
# https://physionet.org/content/mitdb/1.0.0/
# https://physionet.org/content/edb/1.0.0/
# https://physionet.org/content/nstdb/1.0.0/

# ====================================================================================================================================== #
# PREPROCESSING: Reading the signal record, extracting all the features and corresponding labels, normalizing the features, creating a   #                                                                     #
# corresponding array X and Y to store these values in.                                                                                  #
# ====================================================================================================================================== #
import numpy as np
import matplotlib.pyplot as plt
import wfdb
from wfdb import processing, rdann
import os

X = [] # Initialize an empty list to store features (segmented ECG beats).
Y = [] # Initialize an empty list to store labels (annotations).

'''
MIT/BIH records
'''
records = [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 111, 112, 113, 114, 115,
           116, 117, 118, 119, 121, 122, 123, 124, 200, 201, 202, 203, 205, 207, 208, 
           209, 210, 212, 213, 214, 215, 217, 219, 220, 221, 222, 223, 228, 230, 231,
           232, 233, 234]

for record in records:
    ecg_signal, record_info = wfdb.rdsamp(f'C:\\..\\{record}', channels=[0])
    fs = record_info['fs']
    annotation = rdann(f'C:\\..\\{record}', 'atr')
    qrs_locs = annotation.sample 
    labels = annotation.symbol

    # Label adjustment
    for i in range(len(labels)):
        if labels[i] == 'A':
            labels[i] = 'S'
        elif labels[i] == 'Q':
            labels[i] = 'X'
    
    segment_length = int(0.65 * fs)
    segments = [] 
    qrs_num = len(qrs_locs)
    
    for i in range(qrs_num):    
        start_idx = int(qrs_locs[i] - 0.4*fs)
        end_idx = int(qrs_locs[i] + 0.25*fs)
        segment = ecg_signal[start_idx:end_idx]
        if len(segment) != segment_length:
            continue
        if segment.size != 0:
            mean = np.mean(segment) # Sample mean
            std = np.std(segment) # Sample standard deviation
            n_segment = (segment - mean)/std # Normalized segment
            segment_info = {
                'id': i,
                'segment': n_segment,
                'label': labels[i]
            }
            if segment_info['label'] in ['N', 'S', 'X', 'V']:
                segments.append(segment_info)
    
    segments = sorted(segments, key=lambda x: x['id'])
    
    for segment_info in segments: # Of the shape (1,N,1), suitable for RNNs and Conv1D 
        segment_info['segment'] = segment_info['segment'].reshape(segment_length) # Turning the 3D segment array into a 2D segment
        X.append(segment_info['segment']) # Features list
        Y.append(segment_info['label']) # Labels list
    
X = np.array(X)
Y = np.array(Y)

# SMOTE data augmentation only works for classes where we have at least 6 samples
unique_labels, counts = np.unique(Y, return_counts=True)
for element, count in zip(unique_labels, counts):
    if count < 6: 
        indices = np.where(Y == element)[0]
        Y = np.delete(Y, indices)
        X = np.delete(X, indices, axis=0)
        unique_labels, counts = np.unique(Y, return_counts=True)
        print(counts)
        num_classes = len(counts)

# ====================================================================================================================================== #
# MODEL ARCHITECTURE: building the hybrid Neural Network model architecture                                                              #
# ====================================================================================================================================== #
from keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, SeparableConv1D, AveragePooling1D, Dropout, GRU, Concatenate, Dense, LSTM
from keras.models import Model

# Input layer
input_layer = Input(shape=(segment_length, 1))

# Block1
block1_layer1 = Conv1D(filters=32, kernel_size=10, activation='elu', strides=1, padding='same')(input_layer)
block1_pool1 = MaxPooling1D(pool_size=2, strides=2, padding='same')(block1_layer1)
block1_layer2 = Conv1D(filters=64, kernel_size=10, activation='elu', strides=1, padding='same')(block1_pool1)
block1_pool2 = MaxPooling1D(pool_size=2, strides=2, padding='same')(block1_layer2)
block1_global_pool = GlobalAveragePooling1D()(block1_pool2)

# Block2
block2_layer1 = SeparableConv1D(filters=64, kernel_size=15, activation='elu', strides=1, padding='same')(input_layer)
block2_pool1 = AveragePooling1D(pool_size=2, strides=2, padding='same')(block2_layer1)
block2_dropout1 = Dropout(rate=0.2)(block2_pool1)
block2_layer2 = SeparableConv1D(filters=128, kernel_size=15, activation='elu', strides=1, padding='same')(block2_dropout1)
block2_pool2 = AveragePooling1D(pool_size=2, strides=2, padding='same')(block2_layer2)
block2_dropout2 = Dropout(rate=0.2)(block2_pool2)
block2_layer3 = SeparableConv1D(filters=256, kernel_size=15, activation='elu', strides=1, padding='same')(block2_dropout2)
block2_pool3 = AveragePooling1D(pool_size=2, strides=2, padding='same')(block2_layer3)
block2_dropout3 = Dropout(rate=0.2)(block2_pool3)
block2_gru = GRU(units=32)(block2_dropout3)

# Combining the results
R1 = Concatenate()([block1_global_pool, block2_gru])

# Block3
block3_layer1 = Conv1D(filters=64, kernel_size=15, activation='elu', strides=1, padding='same')(input_layer)
block3_pool1 = MaxPooling1D(pool_size=2, strides=2, padding='same')(block3_layer1)
block3_dropout1 = Dropout(rate=0.3)(block3_pool1)
block3_layer2 = Conv1D(filters=128, kernel_size=15, activation='elu', strides=1, padding='same')(block3_dropout1)
block3_pool2 = MaxPooling1D(pool_size=2, strides=2, padding='same')(block3_layer2)
block3_dropout2 = Dropout(rate=0.3)(block3_pool2)
block3_layer3 = Conv1D(filters=256, kernel_size=15, activation='elu', strides=1, padding='same')(block3_dropout2)
block3_pool3 = MaxPooling1D(pool_size=2, strides=2, padding='same')(block3_layer3)
block3_dropout3 = Dropout(rate=0.3)(block3_pool3)
block3_lstm = LSTM(units=32)(block3_dropout3)

# Combining the results
R2 = Concatenate()([block1_global_pool, block3_lstm])

# Combining the combined results
R = Concatenate()([R1, R2])

# Output layer
output_layer = Dense(units=4, activation='softmax')(R)

# Model
# model = Model(inputs=input_layer, outputs=output_layer)

# ====================================================================================================================================== #
# Compiling the model architecture with categorical crossentropy loss function and the Adpative Moment Optimizer                         #
# ====================================================================================================================================== #
from keras import metrics
HCRNet = Model(inputs=input_layer, outputs=output_layer)
HCRNet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=[metrics.Precision(), metrics.Recall()])
HCRNet.summary()

# ====================================================================================================================================== #
# Applying SMOTE, 10-fold cross validation, training and testing the model and outputting the results                                    #                                                  #
# ====================================================================================================================================== #
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

label_encoder = LabelEncoder()
Y_encoded = label_encoder.fit_transform(Y)
kf = StratifiedKFold(n_splits=10)
precision_scores = []
recall_scores = []

for fold, (train_index, test_index) in enumerate(kf.split(X, Y), 1):
    X_train, X_test = X[train_index], X[test_index]
    Y_train, Y_test = Y[train_index], Y[test_index]
    X_train, Y_train = shuffle(X_train, Y_train)

    # Use SMOTE on the entire training dataset
    sm = SMOTE()
    X_train_oversampled, Y_train_oversampled = sm.fit_resample(X_train, Y_train)
    
    # Split the training data into training (80%) and validation (20%)
    X_train, X_val, Y_train, Y_val = train_test_split(X_train_oversampled, Y_train_oversampled, test_size=0.2, random_state=42)
    X_train = X_train.astype('float32')
    X_val = X_val.astype('float32')
    
    Y_train_encoded = label_encoder.transform(Y_train)
    Y_val_encoded = label_encoder.transform(Y_val)
    Y_test_encoded = label_encoder.transform(Y_test)
    
    n_classes = len(label_encoder.classes_)
    
    Y_train = to_categorical(Y_train_encoded, num_classes=n_classes)
    Y_val = to_categorical(Y_val_encoded, num_classes=n_classes)
   
    history = HCRNet.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=10,
        shuffle=True,
        batch_size=16
    )

    # Evaluate the model on the test data only after the last epoch (30) has been completed
    if fold >= 1:
        Y_pred = HCRNet.predict(X_test)
        Y_pred_classes = np.argmax(Y_pred, axis=1)
        precision = precision_score(Y_test_encoded, Y_pred_classes, average='macro')
        recall = recall_score(Y_test_encoded, Y_pred_classes, average='macro')
        ConfusionMatrixDisplay.from_predictions(Y_test_encoded, Y_pred_classes)
        plt.show()
        print('Precision = ', precision * 100, '%')
        print('Recall = ', recall * 100, '%')
        # Append test scores to lists
        precision_scores.append(precision)
        recall_scores.append(recall)

# Calculate mean precision and recall scores
mean_precision = np.mean(precision_scores)
mean_recall = np.mean(recall_scores)

print(f"Mean Precision: {mean_precision}")
print(f"Mean Recall: {mean_recall}")

# ====================================================================================================================================== #
# Saving the model                                                                                                                       #
# ====================================================================================================================================== #
model.save('HCRNet.keras')

# ====================================================================================================================================== #
# Loading the model                                                                                                                       #
# ====================================================================================================================================== #
from keras.models import load_model
loaded_model = load_model('HCRNet1.keras')